# Feature Universality

Test whether features are universal across different inputs: activation
consistency, position independence, context invariance, shared principal
directions, and layer-wise universality.

## Why This Matters

Feature universality investigates the interpretable directions that models use internally. Understanding features — the natural units of neural computation — is essential for moving beyond neuron-level analysis to a true understanding of what models represent.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_universality import (
    feature_activation_consistency, position_independence,
    context_invariance, feature_clustering_across_inputs,
    layer_wise_feature_universality,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens_list = [
    jnp.array([1, 15, 30, 45, 60, 75, 90]),
    jnp.array([5, 20, 35, 50, 65, 80, 95]),
    jnp.array([10, 25, 40, 55, 70, 85, 99]),
]
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
print('Model ready')

## Feature Activation Consistency

Does a feature direction activate consistently across different inputs?

In [ ]:
result = feature_activation_consistency(model, tokens_list, layer=1, direction=direction)
print(f"Consistency score: {result['consistency_score']:.4f}")
print(f"Mean across inputs: {result['mean_across_inputs']:.4f}\n")
for p in result['per_input']:
    print(f"  Input {p['input_idx']}: mean_proj={p['mean_projection']:.4f}, "
          f"max_proj={p['max_projection']:.4f}, active={p['active_fraction']:.2%}")

## Position Independence

Does the feature activate regardless of position?

In [ ]:
result = position_independence(model, tokens_list[0], layer=1, direction=direction)
print(f"Position correlation: {result['position_correlation']:+.4f}")
print(f"Position independent: {result['is_position_independent']}\n")
for p in result['per_position']:
    bar = '█' * max(0, int(p['projection'] * 20 + 10))
    print(f"  Pos {p['position']}: {p['projection']:+.4f} {bar}")

## Context Invariance

Does the feature respond similarly in different contexts?

In [ ]:
result = context_invariance(model, tokens_list[0], tokens_list[1], layer=1, direction=direction)
print(f"Mean projection A: {result['mean_projection_a']:.4f}")
print(f"Mean projection B: {result['mean_projection_b']:.4f}")
print(f"Relative difference: {result['relative_difference']:.4f}")
print(f"Invariant: {result['is_invariant']}")

## Feature Clustering Across Inputs

Find principal directions shared across inputs via joint PCA.

In [ ]:
result = feature_clustering_across_inputs(model, tokens_list, layer=1, n_directions=5)
print(f"Mean consistency: {result['mean_consistency']:.4f}\n")
for d in result['per_direction']:
    print(f"  Direction {d['direction_idx']}: var_explained={d['variance_explained']:.4f}, "
          f"consistency={d['consistency']:.4f}")

## Layer-Wise Feature Universality

At which layers is the feature most universal?

In [ ]:
result = layer_wise_feature_universality(model, tokens_list, direction=direction)
print(f"Universal fraction: {result['universal_fraction']:.2%}")
print(f"Most universal layer: {result['most_universal_layer']}\n")
for p in result['per_layer']:
    status = 'UNIVERSAL' if p['is_universal'] else 'variable'
    print(f"  Layer {p['layer']}: mean={p['mean_projection']:.4f}, "
          f"cv={p['coefficient_of_variation']:.4f} [{status}]")